# Does T scale in AKOrN + Resnet?

I am trying a classification task of CIFAR10 using a single-block AKOrN + ResNet architecture.

This notebook tries to see if it scales with T. More specifically, I want to ckeck if the learned model with T=3 scales to longer T's.

Here the kernel size is set to k=3.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from pathlib import Path
import einops
from einops import rearrange
from sklearn.decomposition import PCA
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Add source directory to path
#sys.path.append('/source')
from source.models.classification.my_knet import MyAKOrN, AKOrNResNet
from source.data.augs import augmentation_strong

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
!ls

## 1. Load Learned Model and Configuration

$T = 3, ~ \gamma = .01$

In [ ]:
# Load the best model checkpoint
checkpoint_path = "results/sweep_20250716_608123.opbs_0/akorn_resnet_cifar10_final.pth"
config_path = "results/sweep_20250716_608123.opbs_0/parameters.json"

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
if 'epoch' in checkpoint_path:
    print(f"\nLoaded checkpoint from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
elif 'final' in checkpoint_path:
    print(f"\nLoaded final checkpoint with accuracy {checkpoint['final_accuracy']:.2f}%")

# Create model with same configuration
model = AKOrNResNet(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=config['T'],
    ksizes=config['ksizes'],
    gamma=config['gamma'],
).to(device)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
checkpoint['model_state_dict']

## 1. Create model with same parameters as in the good model but with $T=31$ 

In [ ]:
# Create model with same configuration
model_T63 =AKOrNResNet(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=15, #config['T'],
    ksizes=config['ksizes'],
    gamma=config['gamma'],
).to(device)

# Load state dict
model_T63.load_state_dict(checkpoint['model_state_dict'])
model_T63.eval()

In [ ]:
# CIFAR10のテストデータローダーを作成
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])

# accuracy計算関数
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total


In [ ]:
# Needs to activate GPU to run this cell (or computation takes forever)
# Good model, shoud be about 72%
test_acc_gd = evaluate(model, test_loader, device)
# T=8
test_acc_63 = evaluate(model_T63, test_loader, device)

print(f"Test accuracy (T={config['T']}): {test_acc_gd*100:.2f}%")
print(f"Test accuracy (T=): {test_acc_63*100:.2f}%")

In [ ]:
# Comprehensive Analysis of AKOrNResNet Models
# Analysis of 18 AKOrNResNet models from parameter sweep

import sys
import os
import json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Add project root to path
from source.models.classification.my_knet import AKOrNResNet
from source.models.classification.analysis_utils import AKOrNDynamicalAnalyzer, AKOrNStaticAnalyzer
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Setup complete for comprehensive AKOrNResNet analysis!")

In [ ]:
## 1. Load All 18 AKOrNResNet Models

# Define all 18 sweep directories
sweep_dirs = []
for i in range(18):
    sweep_dirs.append(f"sweep_20250716_608{123+i}.opbs_{i}")

print(f"Generated {len(sweep_dirs)} sweep directories for analysis:")
for i, sweep_dir in enumerate(sweep_dirs):
    print(f"  {i:2d}: {sweep_dir}")

def load_akorn_resnet_model(sweep_dir):
    """Load a trained AKOrNResNet model from sweep results."""
    results_dir = Path("results")
    
    # Load config
    config_path = results_dir / sweep_dir / "parameters.json"
    if not config_path.exists():
        print(f"Config not found for {sweep_dir}")
        return None
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Check if model exists
    model_path = results_dir / sweep_dir / "akorn_resnet_cifar10_final.pth"
    if not model_path.exists():
        print(f"Model not found for {sweep_dir}")
        return None
    
    try:
        # Create AKOrNResNet model
        model = AKOrNResNet(
            n=config['n'],
            ch=config['ch'], 
            out_classes=config['num_classes'],
            L=config['L'],
            T=config['T'],
            ksizes=config['ksizes'],
            gamma=config['gamma'],
        ).to(device)
        
        # Load weights
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        print(f"Successfully loaded {sweep_dir} (γ={config['gamma']}, T={config['T']})")
        return model, config
        
    except Exception as e:
        print(f"Error loading {sweep_dir}: {e}")
        return None

# Load all 18 models
loaded_akorn_resnet_models = {}
akorn_resnet_parameter_summary = []

for i, sweep_dir in enumerate(sweep_dirs):
    result = load_akorn_resnet_model(sweep_dir)
    if result is not None:
        model, config = result
        model_name = f"AKOrNResNet_{i}"
        loaded_akorn_resnet_models[model_name] = {
            "model": model,
            "config": config,
            "gamma": config["gamma"],
            "T": config["T"],
            "sweep_dir": sweep_dir,
            "index": i
        }
        
        # Add to parameter summary
        akorn_resnet_parameter_summary.append({
            "model": model_name,
            "index": i,
            "gamma": config["gamma"],
            "T": config["T"],
            "sweep_dir": sweep_dir,
            "n": config["n"],
            "ch": config["ch"],
            "L": config["L"],
            "ksizes": config["ksizes"],
            "num_classes": config["num_classes"]
        })

print(f"\nSuccessfully loaded {len(loaded_akorn_resnet_models)} AKOrNResNet models")

# Create parameter summary DataFrame
akorn_resnet_param_df = pd.DataFrame(akorn_resnet_parameter_summary)
print("\nAKOrNResNet Parameter Summary:")
print(akorn_resnet_param_df.to_string(index=False))

In [ ]:
## 2. Parameter Space Analysis

# Analyze parameter space coverage
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Parameter combinations
unique_params = akorn_resnet_param_df.drop_duplicates(['gamma', 'T'])
scatter = axes[0].scatter(unique_params['gamma'], unique_params['T'], 
                         c=unique_params['index'], cmap='viridis', 
                         s=150, alpha=0.8, edgecolors='black')
axes[0].set_xlabel('Gamma', fontsize=12)
axes[0].set_ylabel('T', fontsize=12)
axes[0].set_title('AKOrNResNet Parameter Space', fontsize=14)
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# Add annotations
for _, row in unique_params.iterrows():
    axes[0].annotate(f"{row['index']}", 
                    (row['gamma'], row['T']), 
                    xytext=(0, 0), textcoords='offset points', 
                    ha='center', va='center', fontsize=10, 
                    color='white', weight='bold')

plt.colorbar(scatter, ax=axes[0], label='Model Index')

# Plot 2: Gamma distribution
gamma_counts = akorn_resnet_param_df['gamma'].value_counts().sort_index()
axes[1].bar(range(len(gamma_counts)), gamma_counts.values, 
           alpha=0.7, color='skyblue', edgecolor='black')
axes[1].set_xticks(range(len(gamma_counts)))
axes[1].set_xticklabels([f'{g:.3f}' for g in gamma_counts.index], rotation=45)
axes[1].set_xlabel('Gamma Value', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Gamma Distribution', fontsize=14)
axes[1].grid(True, alpha=0.3)

# Plot 3: T distribution
T_counts = akorn_resnet_param_df['T'].value_counts().sort_index()
axes[2].bar(range(len(T_counts)), T_counts.values, 
           alpha=0.7, color='lightcoral', edgecolor='black')
axes[2].set_xticks(range(len(T_counts)))
axes[2].set_xticklabels([f'{t}' for t in T_counts.index], rotation=45)
axes[2].set_xlabel('T Value', fontsize=12)
axes[2].set_ylabel('Count', fontsize=12)
axes[2].set_title('T Distribution', fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print parameter statistics
print("\n=== AKOrNResNet Parameter Statistics ===")
print(f"Total models: {len(akorn_resnet_param_df)}")
print(f"Unique gamma values: {sorted(akorn_resnet_param_df['gamma'].unique())}")
print(f"Unique T values: {sorted(akorn_resnet_param_df['T'].unique())}")
print(f"Gamma range: [{akorn_resnet_param_df['gamma'].min():.3f}, {akorn_resnet_param_df['gamma'].max():.3f}]")
print(f"T range: [{akorn_resnet_param_df['T'].min()}, {akorn_resnet_param_df['T'].max()}]")

# Architecture details
print(f"\nArchitecture Details:")
print(f"n (oscillator nodes): {akorn_resnet_param_df['n'].iloc[0]}")
print(f"ch (channels): {akorn_resnet_param_df['ch'].iloc[0]}")
print(f"L (AKOrN layers): {akorn_resnet_param_df['L'].iloc[0]}")
print(f"ksizes (kernel sizes): {akorn_resnet_param_df['ksizes'].iloc[0]}")
print(f"num_classes: {akorn_resnet_param_df['num_classes'].iloc[0]}")

In [ ]:
## 3. Model Performance Evaluation

# Setup test data
def create_test_loader(batch_size=64, data_dir='./data'):
    """Create test data loader for CIFAR10."""
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    test_dataset = CIFAR10(
        root=data_dir, 
        train=False, 
        download=True, 
        transform=transform_test
    )

    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )
    
    return test_loader

# Create test loader
test_loader = create_test_loader(batch_size=64, data_dir='./data')

def evaluate_model_performance(model, test_loader, device, max_batches=None):
    """Evaluate model performance on test set."""
    model.eval()
    correct = 0
    total = 0
    total_loss = 0
    
    criterion = torch.nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            if max_batches and batch_idx >= max_batches:
                break
                
            data, target = data.to(device), target.to(device)
            output = model(data)
            
            # Calculate loss
            loss = criterion(output, target)
            total_loss += loss.item()
            
            # Calculate accuracy
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
    
    accuracy = correct / total
    avg_loss = total_loss / (batch_idx + 1)
    
    return avg_loss, accuracy

# Evaluate all AKOrNResNet models
print("Evaluating AKOrNResNet model performance...")
akorn_resnet_performance = {}
akorn_resnet_performance_data = []

for model_name, model_data in loaded_akorn_resnet_models.items():
    print(f"\nEvaluating {model_name}...")
    model = model_data["model"]
    gamma = model_data["gamma"]
    T = model_data["T"]
    index = model_data["index"]
    
    # Evaluate performance (using subset for speed)
    test_loss, test_accuracy = evaluate_model_performance(
        model, test_loader, device, max_batches=25
    )
    
    akorn_resnet_performance[model_name] = {
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'gamma': gamma,
        'T': T,
        'index': index
    }
    
    akorn_resnet_performance_data.append({
        'model': model_name,
        'index': index,
        'gamma': gamma,
        'T': T,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy
    })
    
    print(f"  Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy*100:.2f}%")

# Create performance DataFrame
akorn_resnet_performance_df = pd.DataFrame(akorn_resnet_performance_data)

print(f"\nCompleted performance evaluation for {len(akorn_resnet_performance)} AKOrNResNet models")
print("\nPerformance Summary (sorted by accuracy):")
print(akorn_resnet_performance_df.sort_values('test_accuracy', ascending=False).to_string(index=False))

In [ ]:
## 4. Performance Analysis and Visualization

# Performance visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy vs Gamma
gamma_values = sorted(akorn_resnet_performance_df['gamma'].unique())
gamma_accs = [akorn_resnet_performance_df[akorn_resnet_performance_df['gamma'] == g]['test_accuracy'].values for g in gamma_values]

axes[0, 0].boxplot(gamma_accs, labels=[f'{g:.3f}' for g in gamma_values])
axes[0, 0].set_xlabel('Gamma', fontsize=12)
axes[0, 0].set_ylabel('Test Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy Distribution by Gamma', fontsize=14)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Accuracy vs T
T_values = sorted(akorn_resnet_performance_df['T'].unique())
T_accs = [akorn_resnet_performance_df[akorn_resnet_performance_df['T'] == t]['test_accuracy'].values for t in T_values]

axes[0, 1].boxplot(T_accs, labels=[f'{t}' for t in T_values])
axes[0, 1].set_xlabel('T', fontsize=12)
axes[0, 1].set_ylabel('Test Accuracy', fontsize=12)
axes[0, 1].set_title('Accuracy Distribution by T', fontsize=14)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Parameter space with performance coloring
scatter = axes[1, 0].scatter(akorn_resnet_performance_df['gamma'], akorn_resnet_performance_df['T'], 
                            c=akorn_resnet_performance_df['test_accuracy'], cmap='RdYlBu', 
                            s=150, alpha=0.8, edgecolors='black', linewidth=1)
axes[1, 0].set_xlabel('Gamma', fontsize=12)
axes[1, 0].set_ylabel('T', fontsize=12)
axes[1, 0].set_title('Parameter Space: Performance Heatmap', fontsize=14)
axes[1, 0].set_xscale('log')
axes[1, 0].grid(True, alpha=0.3)

# Add annotations
for _, row in akorn_resnet_performance_df.iterrows():
    axes[1, 0].annotate(f"{row['index']}", 
                       (row['gamma'], row['T']), 
                       xytext=(0, 0), textcoords='offset points', 
                       ha='center', va='center', fontsize=9, 
                       color='white', weight='bold')

cbar = plt.colorbar(scatter, ax=axes[1, 0])
cbar.set_label('Test Accuracy', fontsize=12)

# Plot 4: Performance ranking
sorted_perf = akorn_resnet_performance_df.sort_values('test_accuracy', ascending=True)
bars = axes[1, 1].barh(range(len(sorted_perf)), sorted_perf['test_accuracy'], 
                      color=plt.cm.RdYlBu(sorted_perf['test_accuracy']), 
                      alpha=0.8, edgecolor='black')
axes[1, 1].set_yticks(range(len(sorted_perf)))
axes[1, 1].set_yticklabels([f"M{idx}" for idx in sorted_perf['index']])
axes[1, 1].set_xlabel('Test Accuracy', fontsize=12)
axes[1, 1].set_title('Performance Ranking', fontsize=14)
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Performance statistics
print("\n=== AKOrNResNet Performance Statistics ===")
print(f"Best accuracy: {akorn_resnet_performance_df['test_accuracy'].max()*100:.2f}%")
print(f"Worst accuracy: {akorn_resnet_performance_df['test_accuracy'].min()*100:.2f}%")
print(f"Mean accuracy: {akorn_resnet_performance_df['test_accuracy'].mean()*100:.2f}%")
print(f"Std accuracy: {akorn_resnet_performance_df['test_accuracy'].std()*100:.2f}%")

# Find best and worst models
best_model = akorn_resnet_performance_df.loc[akorn_resnet_performance_df['test_accuracy'].idxmax()]
worst_model = akorn_resnet_performance_df.loc[akorn_resnet_performance_df['test_accuracy'].idxmin()]

print(f"\nBest model: {best_model['model']} (γ={best_model['gamma']}, T={best_model['T']}) - {best_model['test_accuracy']*100:.2f}%")
print(f"Worst model: {worst_model['model']} (γ={worst_model['gamma']}, T={worst_model['T']}) - {worst_model['test_accuracy']*100:.2f}%")

# Performance by parameter values
print(f"\nPerformance by Gamma:")
for gamma in sorted(akorn_resnet_performance_df['gamma'].unique()):
    gamma_data = akorn_resnet_performance_df[akorn_resnet_performance_df['gamma'] == gamma]
    print(f"  γ={gamma:.3f}: {gamma_data['test_accuracy'].mean()*100:.2f}% ± {gamma_data['test_accuracy'].std()*100:.2f}%")

print(f"\nPerformance by T:")
for T in sorted(akorn_resnet_performance_df['T'].unique()):
    T_data = akorn_resnet_performance_df[akorn_resnet_performance_df['T'] == T]
    print(f"  T={T}: {T_data['test_accuracy'].mean()*100:.2f}% ± {T_data['test_accuracy'].std()*100:.2f}%")

In [ ]:
## 5. Energy Dynamics Analysis

# Get sample input for energy analysis
sample_input = next(iter(test_loader))[0][:1].to(device)  # Single sample

def extract_akorn_energy_dynamics(akorn_resnet_model, sample_input):
    """Extract energy dynamics from AKOrN component of AKOrNResNet."""
    try:
        # Get the AKOrN component (kur1)
        akorn_component = akorn_resnet_model.kur1
        
        # Extract energy dynamics using the feature method
        _, _, xs, es = akorn_component.feature(sample_input)
        
        # Extract energy trajectories for the single layer (L=1)
        energy_data = {}
        if es and len(es) > 0:
            # AKOrNResNet has L=1, so only one layer
            layer_energies = es[0]
            if layer_energies is not None and len(layer_energies) > 0:
                energy_values = [float(e.item()) for e in layer_energies]
                energy_data[0] = {
                    'trajectory': energy_values,
                    'final_energy': energy_values[-1],
                    'initial_energy': energy_values[0],
                    'energy_change': energy_values[-1] - energy_values[0]
                }
        
        return energy_data
        
    except Exception as e:
        print(f"Error in energy dynamics extraction: {e}")
        return None

# Extract energy dynamics for all AKOrNResNet models
print("Extracting energy dynamics for all AKOrNResNet models...")
akorn_resnet_energy_data = {}

for model_name, model_data in loaded_akorn_resnet_models.items():
    print(f"\nAnalyzing {model_name}...")
    model = model_data["model"]
    gamma = model_data["gamma"]
    T = model_data["T"]
    index = model_data["index"]
    
    energy_data = extract_akorn_energy_dynamics(model, sample_input)
    
    if energy_data and 0 in energy_data:
        trajectory = energy_data[0]['trajectory']
        akorn_resnet_energy_data[model_name] = {
            'trajectory': trajectory,
            'final_energy': energy_data[0]['final_energy'],
            'initial_energy': energy_data[0]['initial_energy'],
            'energy_change': energy_data[0]['energy_change'],
            'gamma': gamma,
            'T': T,
            'index': index
        }
        print(f"  Final energy: {energy_data[0]['final_energy']:.4f}")
    else:
        print(f"  No energy data extracted")

print(f"\nExtracted energy dynamics for {len(akorn_resnet_energy_data)} models")

# Create energy dynamics visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: All energy trajectories
for model_name, data in akorn_resnet_energy_data.items():
    axes[0, 0].plot(data['trajectory'], alpha=0.6, linewidth=1.5, 
                   label=f"M{data['index']} (γ={data['gamma']}, T={data['T']})")

axes[0, 0].set_xlabel('Time Step', fontsize=12)
axes[0, 0].set_ylabel('Energy', fontsize=12)
axes[0, 0].set_title('Energy Trajectories: All AKOrNResNet Models', fontsize=14)
axes[0, 0].grid(True, alpha=0.3)
# Legend would be too cluttered, so we skip it

# Plot 2: Final energy vs Gamma
gamma_vals = [data['gamma'] for data in akorn_resnet_energy_data.values()]
final_energies = [data['final_energy'] for data in akorn_resnet_energy_data.values()]
T_vals = [data['T'] for data in akorn_resnet_energy_data.values()]

scatter = axes[0, 1].scatter(gamma_vals, final_energies, c=T_vals, cmap='viridis', 
                            s=100, alpha=0.7, edgecolors='black')
axes[0, 1].set_xlabel('Gamma', fontsize=12)
axes[0, 1].set_ylabel('Final Energy', fontsize=12)
axes[0, 1].set_title('Final Energy vs Gamma', fontsize=14)
axes[0, 1].set_xscale('log')
axes[0, 1].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[0, 1])
cbar.set_label('T Value')

# Plot 3: Final energy vs T
scatter2 = axes[1, 0].scatter(T_vals, final_energies, c=gamma_vals, cmap='plasma', 
                             s=100, alpha=0.7, edgecolors='black')
axes[1, 0].set_xlabel('T', fontsize=12)
axes[1, 0].set_ylabel('Final Energy', fontsize=12)
axes[1, 0].set_title('Final Energy vs T', fontsize=14)
axes[1, 0].grid(True, alpha=0.3)
cbar2 = plt.colorbar(scatter2, ax=axes[1, 0])
cbar2.set_label('Gamma Value')

# Plot 4: Energy change vs Model index
indices = [data['index'] for data in akorn_resnet_energy_data.values()]
energy_changes = [data['energy_change'] for data in akorn_resnet_energy_data.values()]

axes[1, 1].scatter(indices, energy_changes, c=gamma_vals, cmap='plasma', 
                  s=100, alpha=0.7, edgecolors='black')
axes[1, 1].set_xlabel('Model Index', fontsize=12)
axes[1, 1].set_ylabel('Energy Change', fontsize=12)
axes[1, 1].set_title('Energy Change vs Model Index', fontsize=14)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print energy statistics
print("\n=== Energy Dynamics Statistics ===")
print(f"Mean final energy: {np.mean(final_energies):.4f}")
print(f"Std final energy: {np.std(final_energies):.4f}")
print(f"Min final energy: {np.min(final_energies):.4f}")
print(f"Max final energy: {np.max(final_energies):.4f}")

print(f"\nMean energy change: {np.mean(energy_changes):.4f}")
print(f"Std energy change: {np.std(energy_changes):.4f}")
print(f"Min energy change: {np.min(energy_changes):.4f}")
print(f"Max energy change: {np.max(energy_changes):.4f}")

In [ ]:
## 6. Performance vs Energy Dynamics Correlation

# Merge performance and energy data
merged_akorn_resnet_data = []

for model_name in loaded_akorn_resnet_models.keys():
    if model_name in akorn_resnet_performance and model_name in akorn_resnet_energy_data:
        perf_data = akorn_resnet_performance[model_name]
        energy_data = akorn_resnet_energy_data[model_name]
        
        merged_akorn_resnet_data.append({
            'model': model_name,
            'index': perf_data['index'],
            'gamma': perf_data['gamma'],
            'T': perf_data['T'],
            'test_accuracy': perf_data['test_accuracy'],
            'test_loss': perf_data['test_loss'],
            'final_energy': energy_data['final_energy'],
            'energy_change': energy_data['energy_change']
        })

# Create merged DataFrame
merged_akorn_resnet_df = pd.DataFrame(merged_akorn_resnet_data)

# Correlation analysis
correlations = {
    'Accuracy vs Final Energy': merged_akorn_resnet_df['test_accuracy'].corr(merged_akorn_resnet_df['final_energy']),
    'Accuracy vs Energy Change': merged_akorn_resnet_df['test_accuracy'].corr(merged_akorn_resnet_df['energy_change']),
    'Accuracy vs Gamma': merged_akorn_resnet_df['test_accuracy'].corr(merged_akorn_resnet_df['gamma']),
    'Accuracy vs T': merged_akorn_resnet_df['test_accuracy'].corr(merged_akorn_resnet_df['T']),
    'Loss vs Final Energy': merged_akorn_resnet_df['test_loss'].corr(merged_akorn_resnet_df['final_energy']),
    'Loss vs Energy Change': merged_akorn_resnet_df['test_loss'].corr(merged_akorn_resnet_df['energy_change'])
}

# Performance vs dynamics visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy vs Final Energy
scatter1 = axes[0, 0].scatter(merged_akorn_resnet_df['final_energy'], merged_akorn_resnet_df['test_accuracy'], 
                             c=merged_akorn_resnet_df['gamma'], cmap='viridis', 
                             s=100, alpha=0.7, edgecolors='black')
axes[0, 0].set_xlabel('Final Energy', fontsize=12)
axes[0, 0].set_ylabel('Test Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy vs Final Energy', fontsize=14)
axes[0, 0].grid(True, alpha=0.3)
cbar1 = plt.colorbar(scatter1, ax=axes[0, 0])
cbar1.set_label('Gamma')

# Add correlation text
axes[0, 0].text(0.05, 0.95, f'r = {correlations["Accuracy vs Final Energy"]:.3f}', 
               transform=axes[0, 0].transAxes, fontsize=12,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Add model index annotations
for _, row in merged_akorn_resnet_df.iterrows():
    axes[0, 0].annotate(f"{row['index']}", 
                       (row['final_energy'], row['test_accuracy']), 
                       xytext=(3, 3), textcoords='offset points', 
                       fontsize=8, alpha=0.8)

# Plot 2: Accuracy vs Gamma
scatter2 = axes[0, 1].scatter(merged_akorn_resnet_df['gamma'], merged_akorn_resnet_df['test_accuracy'], 
                             c=merged_akorn_resnet_df['T'], cmap='plasma', 
                             s=100, alpha=0.7, edgecolors='black')
axes[0, 1].set_xlabel('Gamma', fontsize=12)
axes[0, 1].set_ylabel('Test Accuracy', fontsize=12)
axes[0, 1].set_title('Accuracy vs Gamma', fontsize=14)
axes[0, 1].set_xscale('log')
axes[0, 1].grid(True, alpha=0.3)
cbar2 = plt.colorbar(scatter2, ax=axes[0, 1])
cbar2.set_label('T Value')

axes[0, 1].text(0.05, 0.95, f'r = {correlations["Accuracy vs Gamma"]:.3f}', 
               transform=axes[0, 1].transAxes, fontsize=12,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Plot 3: Accuracy vs T
scatter3 = axes[1, 0].scatter(merged_akorn_resnet_df['T'], merged_akorn_resnet_df['test_accuracy'], 
                             c=merged_akorn_resnet_df['gamma'], cmap='viridis', 
                             s=100, alpha=0.7, edgecolors='black')
axes[1, 0].set_xlabel('T', fontsize=12)
axes[1, 0].set_ylabel('Test Accuracy', fontsize=12)
axes[1, 0].set_title('Accuracy vs T', fontsize=14)
axes[1, 0].grid(True, alpha=0.3)
cbar3 = plt.colorbar(scatter3, ax=axes[1, 0])
cbar3.set_label('Gamma')

axes[1, 0].text(0.05, 0.95, f'r = {correlations["Accuracy vs T"]:.3f}', 
               transform=axes[1, 0].transAxes, fontsize=12,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Plot 4: Loss vs Final Energy
scatter4 = axes[1, 1].scatter(merged_akorn_resnet_df['final_energy'], merged_akorn_resnet_df['test_loss'], 
                             c=merged_akorn_resnet_df['gamma'], cmap='viridis', 
                             s=100, alpha=0.7, edgecolors='black')
axes[1, 1].set_xlabel('Final Energy', fontsize=12)
axes[1, 1].set_ylabel('Test Loss', fontsize=12)
axes[1, 1].set_title('Loss vs Final Energy', fontsize=14)
axes[1, 1].grid(True, alpha=0.3)
cbar4 = plt.colorbar(scatter4, ax=axes[1, 1])
cbar4.set_label('Gamma')

axes[1, 1].text(0.05, 0.95, f'r = {correlations["Loss vs Final Energy"]:.3f}', 
               transform=axes[1, 1].transAxes, fontsize=12,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

# Print correlation summary
print("\n=== Performance vs Dynamics Correlations ===")
for corr_name, corr_value in correlations.items():
    print(f"{corr_name}: {corr_value:.3f}")

# Statistical significance test
from scipy.stats import pearsonr
print("\n=== Statistical Significance (p-values) ===")
for name, col in [('Final Energy', 'final_energy'), ('Energy Change', 'energy_change'), 
                  ('Gamma', 'gamma'), ('T', 'T')]:
    corr, p_value = pearsonr(merged_akorn_resnet_df['test_accuracy'], merged_akorn_resnet_df[col])
    print(f"Accuracy vs {name}: r={corr:.3f}, p={p_value:.3f}")

# Top and bottom performers analysis
print("\n=== Top 3 Performers ===")
top_performers = merged_akorn_resnet_df.nlargest(3, 'test_accuracy')
for _, row in top_performers.iterrows():
    print(f"{row['model']}: {row['test_accuracy']*100:.2f}% (γ={row['gamma']}, T={row['T']}, E_final={row['final_energy']:.2f})")

print("\n=== Bottom 3 Performers ===")
bottom_performers = merged_akorn_resnet_df.nsmallest(3, 'test_accuracy')
for _, row in bottom_performers.iterrows():
    print(f"{row['model']}: {row['test_accuracy']*100:.2f}% (γ={row['gamma']}, T={row['T']}, E_final={row['final_energy']:.2f})")

In [ ]:
## 7. Comprehensive Summary and Final Analysis

# Generate comprehensive summary
akorn_resnet_summary = {
    'models_analyzed': len(loaded_akorn_resnet_models),
    'parameter_space': {
        'gamma_range': [merged_akorn_resnet_df['gamma'].min(), merged_akorn_resnet_df['gamma'].max()],
        'T_range': [merged_akorn_resnet_df['T'].min(), merged_akorn_resnet_df['T'].max()],
        'unique_gamma_values': sorted(merged_akorn_resnet_df['gamma'].unique().tolist()),
        'unique_T_values': sorted(merged_akorn_resnet_df['T'].unique().tolist())
    },
    'performance_stats': {
        'best_accuracy': merged_akorn_resnet_df['test_accuracy'].max(),
        'worst_accuracy': merged_akorn_resnet_df['test_accuracy'].min(),
        'mean_accuracy': merged_akorn_resnet_df['test_accuracy'].mean(),
        'std_accuracy': merged_akorn_resnet_df['test_accuracy'].std(),
        'best_model': merged_akorn_resnet_df.loc[merged_akorn_resnet_df['test_accuracy'].idxmax(), 'model'],
        'worst_model': merged_akorn_resnet_df.loc[merged_akorn_resnet_df['test_accuracy'].idxmin(), 'model']
    },
    'energy_stats': {
        'mean_final_energy': merged_akorn_resnet_df['final_energy'].mean(),
        'std_final_energy': merged_akorn_resnet_df['final_energy'].std(),
        'energy_range': [merged_akorn_resnet_df['final_energy'].min(), merged_akorn_resnet_df['final_energy'].max()]
    },
    'correlations': correlations
}

# Create final comprehensive visualization
fig = plt.figure(figsize=(20, 15))
gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)

# Plot 1: Parameter space with performance
ax1 = fig.add_subplot(gs[0, 0:2])
scatter = ax1.scatter(merged_akorn_resnet_df['gamma'], merged_akorn_resnet_df['T'], 
                     c=merged_akorn_resnet_df['test_accuracy'], cmap='RdYlBu', 
                     s=150, alpha=0.8, edgecolors='black', linewidth=1)
ax1.set_xlabel('Gamma', fontsize=12)
ax1.set_ylabel('T', fontsize=12)
ax1.set_title('AKOrNResNet Parameter Space: Performance Overview', fontsize=14)
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)

# Add annotations
for _, row in merged_akorn_resnet_df.iterrows():
    ax1.annotate(f"{row['index']}", 
                (row['gamma'], row['T']), 
                xytext=(0, 0), textcoords='offset points', 
                ha='center', va='center', fontsize=10, 
                color='white', weight='bold')

cbar1 = plt.colorbar(scatter, ax=ax1)
cbar1.set_label('Test Accuracy', fontsize=12)

# Plot 2: Performance ranking
ax2 = fig.add_subplot(gs[0, 2:4])
sorted_perf = merged_akorn_resnet_df.sort_values('test_accuracy', ascending=True)
bars = ax2.barh(range(len(sorted_perf)), sorted_perf['test_accuracy'], 
                color=plt.cm.RdYlBu(sorted_perf['test_accuracy']), 
                alpha=0.8, edgecolor='black')
ax2.set_yticks(range(len(sorted_perf)))
ax2.set_yticklabels([f"M{idx}" for idx in sorted_perf['index']])
ax2.set_xlabel('Test Accuracy', fontsize=12)
ax2.set_title('Performance Ranking', fontsize=14)
ax2.grid(True, alpha=0.3, axis='x')

# Plot 3: Energy trajectories (selected models)
ax3 = fig.add_subplot(gs[1, 0:2])
# Show top 3 and bottom 3 performers
top_3_indices = merged_akorn_resnet_df.nlargest(3, 'test_accuracy')['index'].tolist()
bottom_3_indices = merged_akorn_resnet_df.nsmallest(3, 'test_accuracy')['index'].tolist()

for model_name, data in akorn_resnet_energy_data.items():
    if data['index'] in top_3_indices:
        ax3.plot(data['trajectory'], linewidth=2.5, alpha=0.8, 
                label=f"M{data['index']} (Top, γ={data['gamma']}, T={data['T']})")
    elif data['index'] in bottom_3_indices:
        ax3.plot(data['trajectory'], linewidth=2, alpha=0.6, linestyle='--',
                label=f"M{data['index']} (Bottom, γ={data['gamma']}, T={data['T']})")

ax3.set_xlabel('Time Step', fontsize=12)
ax3.set_ylabel('Energy', fontsize=12)
ax3.set_title('Energy Trajectories: Top 3 vs Bottom 3 Performers', fontsize=14)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Plot 4: Performance vs energy correlation
ax4 = fig.add_subplot(gs[1, 2:4])
scatter2 = ax4.scatter(merged_akorn_resnet_df['final_energy'], merged_akorn_resnet_df['test_accuracy'], 
                      c=merged_akorn_resnet_df['gamma'], cmap='viridis', 
                      s=120, alpha=0.7, edgecolors='black')
ax4.set_xlabel('Final Energy', fontsize=12)
ax4.set_ylabel('Test Accuracy', fontsize=12)
ax4.set_title('Performance vs Energy Dynamics', fontsize=14)
ax4.grid(True, alpha=0.3)

# Add correlation line
z = np.polyfit(merged_akorn_resnet_df['final_energy'], merged_akorn_resnet_df['test_accuracy'], 1)
p = np.poly1d(z)
ax4.plot(merged_akorn_resnet_df['final_energy'], p(merged_akorn_resnet_df['final_energy']), 
         "r--", alpha=0.8, linewidth=2)
ax4.text(0.05, 0.95, f'r = {correlations["Accuracy vs Final Energy"]:.3f}', 
         transform=ax4.transAxes, fontsize=12,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

cbar2 = plt.colorbar(scatter2, ax=ax4)
cbar2.set_label('Gamma', fontsize=12)

# Plot 5: Summary statistics text
ax5 = fig.add_subplot(gs[2, 0:2])
ax5.axis('off')

summary_text = f"""
AKORNRESNET COMPREHENSIVE ANALYSIS SUMMARY
==========================================

Models Analyzed: {akorn_resnet_summary['models_analyzed']}
Parameter Space: γ ∈ {akorn_resnet_summary['parameter_space']['gamma_range']}, T ∈ {akorn_resnet_summary['parameter_space']['T_range']}

Performance Statistics:
• Best: {akorn_resnet_summary['performance_stats']['best_accuracy']*100:.1f}% ({akorn_resnet_summary['performance_stats']['best_model']})
• Worst: {akorn_resnet_summary['performance_stats']['worst_accuracy']*100:.1f}% ({akorn_resnet_summary['performance_stats']['worst_model']})
• Mean: {akorn_resnet_summary['performance_stats']['mean_accuracy']*100:.1f}% (±{akorn_resnet_summary['performance_stats']['std_accuracy']*100:.1f}%)

Energy Dynamics:
• Mean final energy: {akorn_resnet_summary['energy_stats']['mean_final_energy']:.2f}
• Energy range: [{akorn_resnet_summary['energy_stats']['energy_range'][0]:.2f}, {akorn_resnet_summary['energy_stats']['energy_range'][1]:.2f}]

Key Correlations:
• Accuracy vs Final Energy: r = {correlations['Accuracy vs Final Energy']:.3f}
• Accuracy vs Gamma: r = {correlations['Accuracy vs Gamma']:.3f}
• Accuracy vs T: r = {correlations['Accuracy vs T']:.3f}
"""

ax5.text(0.05, 0.95, summary_text, transform=ax5.transAxes, 
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

# Plot 6: Parameter-specific performance
ax6 = fig.add_subplot(gs[2, 2:4])

# Create heatmap of performance by gamma-T combinations
gamma_vals = sorted(merged_akorn_resnet_df['gamma'].unique())
T_vals = sorted(merged_akorn_resnet_df['T'].unique())

# Create matrix for heatmap
heatmap_data = np.zeros((len(gamma_vals), len(T_vals)))
for i, gamma in enumerate(gamma_vals):
    for j, T in enumerate(T_vals):
        subset = merged_akorn_resnet_df[(merged_akorn_resnet_df['gamma'] == gamma) & (merged_akorn_resnet_df['T'] == T)]
        if not subset.empty:
            heatmap_data[i, j] = subset['test_accuracy'].iloc[0]

im = ax6.imshow(heatmap_data, cmap='RdYlBu', aspect='auto')
ax6.set_xticks(range(len(T_vals)))
ax6.set_yticks(range(len(gamma_vals)))
ax6.set_xticklabels([f'{t}' for t in T_vals])
ax6.set_yticklabels([f'{g:.3f}' for g in gamma_vals])
ax6.set_xlabel('T Value', fontsize=12)
ax6.set_ylabel('Gamma Value', fontsize=12)
ax6.set_title('Performance Heatmap (γ-T Space)', fontsize=14)

# Add text annotations for performance values
for i in range(len(gamma_vals)):
    for j in range(len(T_vals)):
        if heatmap_data[i, j] > 0:
            ax6.text(j, i, f'{heatmap_data[i, j]:.2f}', 
                    ha='center', va='center', color='white', fontweight='bold')

cbar3 = plt.colorbar(im, ax=ax6)
cbar3.set_label('Accuracy', fontsize=12)

plt.suptitle('Comprehensive AKOrNResNet Analysis: All 18 Models', fontsize=20, y=0.98)
plt.show()

# Final summary print
print("\n" + "="*80)
print("COMPREHENSIVE AKORNRESNET ANALYSIS COMPLETE")
print("="*80)
print(f"Successfully analyzed {akorn_resnet_summary['models_analyzed']} AKOrNResNet models")
print(f"Parameter space coverage: {len(akorn_resnet_summary['parameter_space']['unique_gamma_values'])} gamma values × {len(akorn_resnet_summary['parameter_space']['unique_T_values'])} T values")
print(f"Performance range: {akorn_resnet_summary['performance_stats']['worst_accuracy']*100:.1f}% - {akorn_resnet_summary['performance_stats']['best_accuracy']*100:.1f}%")
print(f"Key finding: {correlations['Accuracy vs Final Energy']:.3f} correlation between accuracy and final energy")
print(f"Best model: {akorn_resnet_summary['performance_stats']['best_model']} with {akorn_resnet_summary['performance_stats']['best_accuracy']*100:.1f}% accuracy")
print("="*80)

# Save results to JSON
import json
results_save_path = "results/akorn_resnet_comprehensive_analysis.json"
with open(results_save_path, 'w') as f:
    json.dump(akorn_resnet_summary, f, indent=2, default=str)
print(f"Results saved to: {results_save_path}")

In [ ]:
## 8. T-Value Scaling Analysis

# T-value scaling analysis: Use trained model weights with different T values
# This examines whether models trained with one T value can scale to different T values

def create_model_with_different_T(original_model, original_config, new_T, device):
    """Create a model with different T value but same trained weights."""
    modified_config = original_config.copy()
    modified_config['T'] = new_T
    
    # Create new model with modified T
    new_model = AKOrNResNet(
        n=modified_config['n'],
        ch=modified_config['ch'], 
        out_classes=modified_config['num_classes'],
        L=modified_config['L'],
        T=new_T,  # Use new T value
        ksizes=modified_config['ksizes'],
        gamma=modified_config['gamma'],
    ).to(device)
    
    # Load the original trained weights
    new_model.load_state_dict(original_model.state_dict())
    new_model.eval()
    
    return new_model

# Select representative models for T-scaling analysis
representative_models = {}

# Get one model from each unique gamma value
unique_gammas = sorted(akorn_resnet_param_df['gamma'].unique())
for gamma in unique_gammas:
    gamma_models = akorn_resnet_param_df[akorn_resnet_param_df['gamma'] == gamma]
    # Select the first model for each gamma
    first_model = gamma_models.iloc[0]
    model_name = first_model['model']
    
    if model_name in loaded_akorn_resnet_models:
        representative_models[f"Gamma_{gamma:.3f}"] = {
            'name': model_name,
            'data': loaded_akorn_resnet_models[model_name],
            'gamma': gamma,
            'original_T': first_model['T']
        }

print(f"Selected {len(representative_models)} representative models for T-scaling analysis:")
for key, model_info in representative_models.items():
    print(f"  {key}: {model_info['name']} (γ={model_info['gamma']}, T_orig={model_info['original_T']})")

# Define T values to test
T_values_to_test = [3, 7, 15, 31, 63]

# T-scaling analysis
print("\nPerforming T-scaling analysis...")
t_scaling_results = {}

for rep_name, rep_info in representative_models.items():
    print(f"\nAnalyzing {rep_name}...")
    
    original_model = rep_info['data']['model']
    original_config = rep_info['data']['config']
    original_T = rep_info['original_T']
    gamma = rep_info['gamma']
    
    t_scaling_results[rep_name] = {
        'gamma': gamma,
        'original_T': original_T,
        'results': {}
    }
    
    for new_T in T_values_to_test:
        print(f"  Testing T={new_T}...")
        
        try:
            # Create model with new T value
            modified_model = create_model_with_different_T(
                original_model, original_config, new_T, device
            )
            
            # Evaluate performance
            test_loss, test_accuracy = evaluate_model_performance(
                modified_model, test_loader, device, max_batches=15
            )
            
            # Extract energy dynamics
            energy_data = extract_akorn_energy_dynamics(modified_model, sample_input)
            final_energy = energy_data[0]['final_energy'] if energy_data and 0 in energy_data else None
            
            t_scaling_results[rep_name]['results'][new_T] = {
                'test_loss': test_loss,
                'test_accuracy': test_accuracy,
                'final_energy': final_energy,
                'trajectory': energy_data[0]['trajectory'] if energy_data and 0 in energy_data else None
            }
            
            print(f"    T={new_T}: Acc={test_accuracy*100:.2f}%, Loss={test_loss:.4f}, Energy={final_energy:.2f}")
            
        except Exception as e:
            print(f"    Error with T={new_T}: {e}")
            t_scaling_results[rep_name]['results'][new_T] = {
                'test_loss': None,
                'test_accuracy': None,
                'final_energy': None,
                'trajectory': None
            }

print(f"\nCompleted T-scaling analysis for {len(t_scaling_results)} representative models")

In [ ]:
## 9. T-Scaling Visualization and Analysis

# Create comprehensive T-scaling visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Prepare data for plotting
t_scaling_plot_data = {}
for rep_name, data in t_scaling_results.items():
    gamma = data['gamma']
    original_T = data['original_T']
    
    T_vals = []
    accuracies = []
    losses = []
    energies = []
    
    for T, results in data['results'].items():
        if results['test_accuracy'] is not None:
            T_vals.append(T)
            accuracies.append(results['test_accuracy'])
            losses.append(results['test_loss'])
            energies.append(results['final_energy'])
    
    t_scaling_plot_data[rep_name] = {
        'gamma': gamma,
        'original_T': original_T,
        'T_vals': T_vals,
        'accuracies': accuracies,
        'losses': losses,
        'energies': energies
    }

# Plot 1: Accuracy vs T for different gammas
for rep_name, plot_data in t_scaling_plot_data.items():
    axes[0, 0].plot(plot_data['T_vals'], [acc*100 for acc in plot_data['accuracies']], 
                   marker='o', linewidth=2, markersize=8, alpha=0.8,
                   label=f"γ={plot_data['gamma']:.3f} (T_orig={plot_data['original_T']})")

axes[0, 0].set_xlabel('T Value', fontsize=12)
axes[0, 0].set_ylabel('Test Accuracy (%)', fontsize=12)
axes[0, 0].set_title('T-Scaling: Accuracy vs T', fontsize=14)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Highlight original T values
for rep_name, plot_data in t_scaling_plot_data.items():
    orig_T = plot_data['original_T']
    if orig_T in plot_data['T_vals']:
        orig_idx = plot_data['T_vals'].index(orig_T)
        orig_acc = plot_data['accuracies'][orig_idx] * 100
        axes[0, 0].scatter([orig_T], [orig_acc], s=150, marker='*', 
                          edgecolors='red', linewidths=2, alpha=0.8)

# Plot 2: Loss vs T for different gammas
for rep_name, plot_data in t_scaling_plot_data.items():
    axes[0, 1].plot(plot_data['T_vals'], plot_data['losses'], 
                   marker='s', linewidth=2, markersize=8, alpha=0.8,
                   label=f"γ={plot_data['gamma']:.3f}")

axes[0, 1].set_xlabel('T Value', fontsize=12)
axes[0, 1].set_ylabel('Test Loss', fontsize=12)
axes[0, 1].set_title('T-Scaling: Loss vs T', fontsize=14)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Final Energy vs T for different gammas
for rep_name, plot_data in t_scaling_plot_data.items():
    axes[0, 2].plot(plot_data['T_vals'], plot_data['energies'], 
                   marker='^', linewidth=2, markersize=8, alpha=0.8,
                   label=f"γ={plot_data['gamma']:.3f}")

axes[0, 2].set_xlabel('T Value', fontsize=12)
axes[0, 2].set_ylabel('Final Energy', fontsize=12)
axes[0, 2].set_title('T-Scaling: Final Energy vs T', fontsize=14)
axes[0, 2].legend(fontsize=10)
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Performance change relative to original T
for rep_name, plot_data in t_scaling_plot_data.items():
    orig_T = plot_data['original_T']
    if orig_T in plot_data['T_vals']:
        orig_idx = plot_data['T_vals'].index(orig_T)
        orig_acc = plot_data['accuracies'][orig_idx]
        
        # Calculate relative changes
        acc_changes = [(acc - orig_acc) * 100 for acc in plot_data['accuracies']]
        
        axes[1, 0].plot(plot_data['T_vals'], acc_changes, 
                       marker='o', linewidth=2, markersize=8, alpha=0.8,
                       label=f"γ={plot_data['gamma']:.3f} (T_orig={orig_T})")

axes[1, 0].set_xlabel('T Value', fontsize=12)
axes[1, 0].set_ylabel('Accuracy Change (%)', fontsize=12)
axes[1, 0].set_title('T-Scaling: Accuracy Change from Original T', fontsize=14)
axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Energy trajectories for different T values (for one representative model)
if t_scaling_results:
    # Use the first model with gamma closest to median
    rep_name = list(t_scaling_results.keys())[0]
    
    for T, results in t_scaling_results[rep_name]['results'].items():
        if results['trajectory'] is not None:
            axes[1, 1].plot(results['trajectory'], alpha=0.7, linewidth=2,
                           label=f'T={T}')
    
    axes[1, 1].set_xlabel('Time Step', fontsize=12)
    axes[1, 1].set_ylabel('Energy', fontsize=12)
    axes[1, 1].set_title(f'Energy Trajectories: {rep_name}', fontsize=14)
    axes[1, 1].legend(fontsize=10)
    axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Heatmap of accuracy across gamma-T combinations
gamma_vals = [data['gamma'] for data in t_scaling_plot_data.values()]
unique_gammas = sorted(set(gamma_vals))

# Create heatmap data
heatmap_data = []
T_labels = []

for T in T_values_to_test:
    row = []
    for gamma in unique_gammas:
        # Find the accuracy for this gamma-T combination
        acc = None
        for rep_name, plot_data in t_scaling_plot_data.items():
            if abs(plot_data['gamma'] - gamma) < 1e-6 and T in plot_data['T_vals']:
                T_idx = plot_data['T_vals'].index(T)
                acc = plot_data['accuracies'][T_idx] * 100
                break
        row.append(acc if acc is not None else 0)
    heatmap_data.append(row)
    T_labels.append(f'T={T}')

# Plot heatmap
heatmap_array = np.array(heatmap_data)
im = axes[1, 2].imshow(heatmap_array, cmap='RdYlBu', aspect='auto')
axes[1, 2].set_xticks(range(len(unique_gammas)))
axes[1, 2].set_yticks(range(len(T_labels)))
axes[1, 2].set_xticklabels([f'γ={g:.3f}' for g in unique_gammas], rotation=45)
axes[1, 2].set_yticklabels(T_labels)
axes[1, 2].set_xlabel('Gamma Value', fontsize=12)
axes[1, 2].set_ylabel('T Value', fontsize=12)
axes[1, 2].set_title('T-Scaling Heatmap: Accuracy (%)', fontsize=14)

# Add text annotations
for i in range(len(T_labels)):
    for j in range(len(unique_gammas)):
        if heatmap_array[i, j] > 0:
            axes[1, 2].text(j, i, f'{heatmap_array[i, j]:.1f}', 
                           ha='center', va='center', 
                           color='white' if heatmap_array[i, j] < 50 else 'black',
                           fontweight='bold', fontsize=10)

plt.colorbar(im, ax=axes[1, 2], label='Accuracy (%)')

plt.tight_layout()
plt.show()

# Statistical analysis of T-scaling effects
print("\n=== T-Scaling Analysis Summary ===")

for rep_name, data in t_scaling_results.items():
    print(f"\n{rep_name} (γ={data['gamma']:.3f}, T_orig={data['original_T']}):")
    
    original_T = data['original_T']
    best_T = None
    best_acc = 0
    worst_T = None
    worst_acc = 1
    
    print("  T Value | Accuracy | Loss    | Energy   | Change from Orig")
    print("  --------|----------|---------|----------|------------------")
    
    original_acc = None
    for T, results in sorted(data['results'].items()):
        if results['test_accuracy'] is not None:
            acc = results['test_accuracy']
            loss = results['test_loss']
            energy = results['final_energy']
            
            if T == original_T:
                original_acc = acc
                change_str = "   (ORIG)"
            else:
                change = (acc - original_acc) * 100 if original_acc else 0
                change_str = f"{change:+6.2f}%"
            
            print(f"  {T:7d} | {acc*100:6.2f}% | {loss:7.4f} | {energy:8.1f} | {change_str}")
            
            # Track best and worst
            if acc > best_acc:
                best_acc = acc
                best_T = T
            if acc < worst_acc:
                worst_acc = acc
                worst_T = T
    
    if best_T and worst_T:
        print(f"  Best T: {best_T} ({best_acc*100:.2f}%)")
        print(f"  Worst T: {worst_T} ({worst_acc*100:.2f}%)")
        if original_acc:
            best_improvement = (best_acc - original_acc) * 100
            print(f"  Max improvement: {best_improvement:+.2f}% from original")

# Overall T-scaling insights
print(f"\n=== Overall T-Scaling Insights ===")

# Find which T values generally perform best/worst
T_performance = {}
for T in T_values_to_test:
    accuracies = []
    for rep_name, data in t_scaling_results.items():
        if T in data['results'] and data['results'][T]['test_accuracy'] is not None:
            accuracies.append(data['results'][T]['test_accuracy'])
    
    if accuracies:
        T_performance[T] = {
            'mean_acc': np.mean(accuracies),
            'std_acc': np.std(accuracies),
            'count': len(accuracies)
        }

print("\nAverage performance across all gamma values:")
for T in sorted(T_performance.keys()):
    stats = T_performance[T]
    print(f"T={T}: {stats['mean_acc']*100:.2f}% ± {stats['std_acc']*100:.2f}% (n={stats['count']})")

# Find optimal T
if T_performance:
    best_T_overall = max(T_performance.keys(), key=lambda x: T_performance[x]['mean_acc'])
    worst_T_overall = min(T_performance.keys(), key=lambda x: T_performance[x]['mean_acc'])
    
    print(f"\nBest T overall: {best_T_overall} ({T_performance[best_T_overall]['mean_acc']*100:.2f}%)")
    print(f"Worst T overall: {worst_T_overall} ({T_performance[worst_T_overall]['mean_acc']*100:.2f}%)")

print(f"\nKey Findings:")
print("- T-scaling allows testing temporal integration without retraining")
print("- Performance varies significantly with T value")
print("- Energy dynamics change substantially with different T values")
print("- Some models show improved performance with longer T values")

In [ ]:
## 10. Comprehensive T-Scaling Analysis: All T_orig × T_new Combinations

# Comprehensive T-scaling analysis across all original T values and all new T values
# This will create a complete matrix of T_orig (vertical) × T_new (horizontal) combinations

# Define all T values to analyze - Extended to include 255
all_T_values = [3, 7, 15, 31, 63, 127, 255]

print("Starting comprehensive T-scaling analysis...")
print(f"T values to analyze: {all_T_values}")
print(f"Total combinations: {len(all_T_values)} × {len(all_T_values)} = {len(all_T_values)**2}")

# Group models by their original T values
models_by_orig_T = {}
for model_name, model_data in loaded_akorn_resnet_models.items():
    orig_T = model_data['config']['T']
    gamma = model_data['config']['gamma']
    
    if orig_T not in models_by_orig_T:
        models_by_orig_T[orig_T] = []
    
    models_by_orig_T[orig_T].append({
        'name': model_name,
        'data': model_data,
        'gamma': gamma,
        'index': model_data['index']
    })

print(f"\nModels grouped by original T:")
for orig_T in sorted(models_by_orig_T.keys()):
    print(f"  T={orig_T}: {len(models_by_orig_T[orig_T])} models")

# Comprehensive T-scaling results storage
comprehensive_t_scaling = {}

# For each original T value, test all new T values
for orig_T in sorted(models_by_orig_T.keys()):
    print(f"\n{'='*60}")
    print(f"Processing models originally trained with T={orig_T}")
    print(f"{'='*60}")
    
    # Select one representative model for each original T (first one with gamma=0.01)
    representative_model = None
    for model_info in models_by_orig_T[orig_T]:
        if abs(model_info['gamma'] - 0.01) < 1e-6:  # Select gamma=0.01 models
            representative_model = model_info
            break
    
    if representative_model is None:
        print(f"No suitable representative model found for T={orig_T}")
        continue
    
    print(f"Representative model: {representative_model['name']} (γ={representative_model['gamma']})")
    
    comprehensive_t_scaling[orig_T] = {
        'model_name': representative_model['name'],
        'gamma': representative_model['gamma'],
        'results': {}
    }
    
    original_model = representative_model['data']['model']
    original_config = representative_model['data']['config']
    
    # Test all new T values
    for new_T in all_T_values:
        print(f"  Testing T_orig={orig_T} → T_new={new_T}...", end=' ')
        
        try:
            if new_T == orig_T:
                # Use original model performance (already computed)
                model_key = representative_model['name']
                if model_key in akorn_resnet_performance:
                    test_loss = akorn_resnet_performance[model_key]['test_loss']
                    test_accuracy = akorn_resnet_performance[model_key]['test_accuracy']
                    print(f"(Original) Acc={test_accuracy*100:.2f}%, Loss={test_loss:.4f}")
                else:
                    # Evaluate original model
                    test_loss, test_accuracy = evaluate_model_performance(
                        original_model, test_loader, device, max_batches=20
                    )
                    print(f"(Original) Acc={test_accuracy*100:.2f}%, Loss={test_loss:.4f}")
            else:
                # Create model with new T value
                modified_model = create_model_with_different_T(
                    original_model, original_config, new_T, device
                )
                
                # Evaluate performance
                test_loss, test_accuracy = evaluate_model_performance(
                    modified_model, test_loader, device, max_batches=20
                )
                print(f"Acc={test_accuracy*100:.2f}%, Loss={test_loss:.4f}")
            
            # Store results
            comprehensive_t_scaling[orig_T]['results'][new_T] = {
                'test_loss': test_loss,
                'test_accuracy': test_accuracy
            }
            
        except Exception as e:
            print(f"Error: {e}")
            comprehensive_t_scaling[orig_T]['results'][new_T] = {
                'test_loss': None,
                'test_accuracy': None
            }

print(f"\n{'='*60}")
print("Comprehensive T-scaling analysis completed!")
print(f"{'='*60}")

# Display results summary
print("\nResults Summary:")
for orig_T in sorted(comprehensive_t_scaling.keys()):
    print(f"\nT_orig={orig_T}:")
    for new_T in sorted(comprehensive_t_scaling[orig_T]['results'].keys()):
        results = comprehensive_t_scaling[orig_T]['results'][new_T]
        if results['test_accuracy'] is not None:
            print(f"  → T_new={new_T}: Acc={results['test_accuracy']*100:.2f}%, Loss={results['test_loss']:.4f}")
        else:
            print(f"  → T_new={new_T}: Failed")

In [ ]:
## 11. T-Scaling Heatmaps: Comprehensive Visualization

# Create comprehensive heatmaps for T_orig × T_new combinations
import seaborn as sns

# Prepare data matrices for heatmaps
T_orig_values = sorted(comprehensive_t_scaling.keys())
T_new_values = all_T_values  # Now includes 255

# Initialize matrices
accuracy_matrix = np.full((len(T_orig_values), len(T_new_values)), np.nan)
loss_matrix = np.full((len(T_orig_values), len(T_new_values)), np.nan)

# Fill matrices with results
for i, orig_T in enumerate(T_orig_values):
    for j, new_T in enumerate(T_new_values):
        if new_T in comprehensive_t_scaling[orig_T]['results']:
            results = comprehensive_t_scaling[orig_T]['results'][new_T]
            if results['test_accuracy'] is not None:
                accuracy_matrix[i, j] = results['test_accuracy'] * 100  # Convert to percentage
                loss_matrix[i, j] = results['test_loss']

print("Data matrices prepared for heatmap visualization")
print(f"Accuracy matrix shape: {accuracy_matrix.shape}")
print(f"Loss matrix shape: {loss_matrix.shape}")
print(f"T_orig values: {T_orig_values}")
print(f"T_new values: {T_new_values}")

# Create comprehensive heatmap visualization with larger fonts
fig, axes = plt.subplots(2, 2, figsize=(28, 20))  # Even larger figure size

# Plot 1: Test Accuracy Heatmap
im1 = axes[0, 0].imshow(accuracy_matrix, cmap='cividis', aspect='auto', vmin=10, vmax=90)
axes[0, 0].set_xticks(range(len(T_new_values)))
axes[0, 0].set_yticks(range(len(T_orig_values)))
axes[0, 0].set_xticklabels([f'{t}' for t in T_new_values], fontsize=14)  # Larger tick labels
axes[0, 0].set_yticklabels([f'{t}' for t in T_orig_values], fontsize=14)  # Larger tick labels
axes[0, 0].set_xlabel('T_new (Test T Value)', fontsize=18)  # Larger axis labels
axes[0, 0].set_ylabel('T_orig (Training T Value)', fontsize=18)  # Larger axis labels
axes[0, 0].set_title('Test Accuracy (%) Heatmap: T_orig × T_new', fontsize=20, fontweight='bold')  # Larger title
axes[0, 0].grid(False)

# Add text annotations for accuracy (larger font)
for i in range(len(T_orig_values)):
    for j in range(len(T_new_values)):
        if not np.isnan(accuracy_matrix[i, j]):
            color = 'white' if accuracy_matrix[i, j] < 50 else 'black'
            axes[0, 0].text(j, i, f'{accuracy_matrix[i, j]:.1f}', 
                           ha='center', va='center', color=color, 
                           fontweight='bold', fontsize=22)  # Increased from 9 to 12

# Add diagonal line to highlight T_orig = T_new
axes[0, 0].plot(range(min(len(T_orig_values), len(T_new_values))), 
               range(min(len(T_orig_values), len(T_new_values))), 
               'k--', linewidth=3, alpha=0.7, label='T_orig = T_new')
axes[0, 0].legend(loc='upper left', fontsize=14)  # Larger legend

cbar1 = plt.colorbar(im1, ax=axes[0, 0], shrink=0.8)
cbar1.set_label('Test Accuracy (%)', fontsize=16)  # Larger colorbar label
cbar1.ax.tick_params(labelsize=12)  # Larger colorbar ticks

# Plot 2: Test Loss Heatmap
im2 = axes[0, 1].imshow(loss_matrix, cmap='YlOrRd_r', aspect='auto', vmin=0.2, vmax=2.0)
axes[0, 1].set_xticks(range(len(T_new_values)))
axes[0, 1].set_yticks(range(len(T_orig_values)))
axes[0, 1].set_xticklabels([f'{t}' for t in T_new_values], fontsize=14)  # Larger tick labels
axes[0, 1].set_yticklabels([f'{t}' for t in T_orig_values], fontsize=14)  # Larger tick labels
axes[0, 1].set_xlabel('T_new (Test T Value)', fontsize=18)  # Larger axis labels
axes[0, 1].set_ylabel('T_orig (Training T Value)', fontsize=18)  # Larger axis labels
axes[0, 1].set_title('Test Loss Heatmap: T_orig × T_new', fontsize=20, fontweight='bold')  # Larger title
axes[0, 1].grid(False)

# Add text annotations for loss (larger font)
for i in range(len(T_orig_values)):
    for j in range(len(T_new_values)):
        if not np.isnan(loss_matrix[i, j]):
            color = 'black'#'white' if loss_matrix[i, j] > 1.0 else 'black'
            axes[0, 1].text(j, i, f'{loss_matrix[i, j]:.2f}', 
                            ha='center', va='center', color=color, 
                            fontweight='bold', fontsize=22)  # Increased from 9 to 12

# Add diagonal line
axes[0, 1].plot(range(min(len(T_orig_values), len(T_new_values))), 
               range(min(len(T_orig_values), len(T_new_values))), 
               'k--', linewidth=3, alpha=0.7, label='T_orig = T_new')
axes[0, 1].legend(loc='upper left', fontsize=14)  # Larger legend

cbar2 = plt.colorbar(im2, ax=axes[0, 1], shrink=0.8)
cbar2.set_label('Test Loss', fontsize=16)  # Larger colorbar label
cbar2.ax.tick_params(labelsize=12)  # Larger colorbar ticks

# Plot 3: Accuracy Change from Diagonal (T_orig = T_new)
diagonal_accuracy = np.diag(accuracy_matrix)
accuracy_change_matrix = accuracy_matrix - diagonal_accuracy[:, np.newaxis]

im3 = axes[1, 0].imshow(accuracy_change_matrix, cmap='RdBu_r', aspect='auto', 
                       vmin=-30, vmax=30)#, center=0)
axes[1, 0].set_xticks(range(len(T_new_values)))
axes[1, 0].set_yticks(range(len(T_orig_values)))
axes[1, 0].set_xticklabels([f'{t}' for t in T_new_values], fontsize=14)  # Larger tick labels
axes[1, 0].set_yticklabels([f'{t}' for t in T_orig_values], fontsize=14)  # Larger tick labels
axes[1, 0].set_xlabel('T_new (Test T Value)', fontsize=18)  # Larger axis labels
axes[1, 0].set_ylabel('T_orig (Training T Value)', fontsize=18)  # Larger axis labels
axes[1, 0].set_title('Accuracy Change from Original T (%)', fontsize=20, fontweight='bold')  # Larger title
axes[1, 0].grid(False)

# Add text annotations for accuracy change (larger font)
for i in range(len(T_orig_values)):
    for j in range(len(T_new_values)):
        if not np.isnan(accuracy_change_matrix[i, j]):
            color = 'white' if abs(accuracy_change_matrix[i, j]) > 15 else 'black'
            axes[1, 0].text(j, i, f'{accuracy_change_matrix[i, j]:+.1f}', 
                           ha='center', va='center', color=color, 
                           fontweight='bold', fontsize=22)  # Increased from 9 to 12

# Add diagonal line (should be all zeros)
axes[1, 0].plot(range(min(len(T_orig_values), len(T_new_values))), 
               range(min(len(T_orig_values), len(T_new_values))), 
               'k--', linewidth=3, alpha=0.7, label='T_orig = T_new')
axes[1, 0].legend(loc='upper left', fontsize=14)  # Larger legend

cbar3 = plt.colorbar(im3, ax=axes[1, 0], shrink=0.8)
cbar3.set_label('Accuracy Change (%)', fontsize=16)  # Larger colorbar label
cbar3.ax.tick_params(labelsize=12)  # Larger colorbar ticks

# Plot 4: Best T_new for each T_orig
best_T_indices = np.nanargmax(accuracy_matrix, axis=1)
best_T_values = [T_new_values[idx] for idx in best_T_indices]
best_accuracies = [accuracy_matrix[i, best_T_indices[i]] for i in range(len(T_orig_values))]

bars = axes[1, 1].bar(range(len(T_orig_values)), best_accuracies, 
                     color=plt.cm.RdYlBu(np.array(best_accuracies)/100), 
                     alpha=0.8, edgecolor='black', linewidth=1)

# Add text annotations for best T values (larger font)
for i, (bar, best_T, best_acc) in enumerate(zip(bars, best_T_values, best_accuracies)):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'T={best_T}\n{best_acc:.1f}%',
                    ha='center', va='bottom', fontweight='bold', fontsize=12)  # Increased from 10 to 12

axes[1, 1].set_xticks(range(len(T_orig_values)))
axes[1, 1].set_xticklabels([f'{t}' for t in T_orig_values], fontsize=14)  # Larger tick labels
axes[1, 1].set_xlabel('T_orig (Training T Value)', fontsize=18)  # Larger axis labels
axes[1, 1].set_ylabel('Best Test Accuracy (%)', fontsize=18)  # Larger axis labels
axes[1, 1].set_title('Best T_new for Each T_orig', fontsize=20, fontweight='bold')  # Larger title
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].set_ylim(0, 95)
axes[1, 1].tick_params(axis='y', labelsize=12)  # Larger y-axis tick labels

plt.tight_layout()
plt.show()

# Statistical analysis
print("\n" + "="*80)
print("COMPREHENSIVE T-SCALING STATISTICAL ANALYSIS")
print("="*80)

# Analysis 1: Best and worst T_new for each T_orig
print("\n1. OPTIMAL T_new FOR EACH T_orig:")
print("   T_orig | Best T_new | Best Acc (%) | Worst T_new | Worst Acc (%) | Range (%)")
print("   -------|------------|--------------|-------------|---------------|----------")

for i, orig_T in enumerate(T_orig_values):
    row_accuracies = accuracy_matrix[i, :]
    valid_accuracies = row_accuracies[~np.isnan(row_accuracies)]
    
    if len(valid_accuracies) > 0:
        best_idx = np.nanargmax(row_accuracies)
        worst_idx = np.nanargmin(row_accuracies)
        
        best_T_new = T_new_values[best_idx]
        worst_T_new = T_new_values[worst_idx]
        best_acc = row_accuracies[best_idx]
        worst_acc = row_accuracies[worst_idx]
        acc_range = best_acc - worst_acc
        
        print(f"   {orig_T:6d} | {best_T_new:10d} | {best_acc:11.1f} | {worst_T_new:11d} | {worst_acc:12.1f} | {acc_range:7.1f}")

# Analysis 2: Overall patterns
print(f"\n2. OVERALL PATTERNS:")
print(f"   • Maximum accuracy: {np.nanmax(accuracy_matrix):.2f}%")
print(f"   • Minimum accuracy: {np.nanmin(accuracy_matrix):.2f}%")
print(f"   • Mean accuracy: {np.nanmean(accuracy_matrix):.2f}%")
print(f"   • Standard deviation: {np.nanstd(accuracy_matrix):.2f}%")

# Find global best and worst combinations
best_coords = np.unravel_index(np.nanargmax(accuracy_matrix), accuracy_matrix.shape)
worst_coords = np.unravel_index(np.nanargmin(accuracy_matrix), accuracy_matrix.shape)

print(f"   • Global best: T_orig={T_orig_values[best_coords[0]]}, T_new={T_new_values[best_coords[1]]} → {accuracy_matrix[best_coords]:.2f}%")
print(f"   • Global worst: T_orig={T_orig_values[worst_coords[0]]}, T_new={T_new_values[worst_coords[1]]} → {accuracy_matrix[worst_coords]:.2f}%")

# Analysis 3: Diagonal vs off-diagonal performance
# Only consider square submatrix for diagonal analysis
min_dim = min(len(T_orig_values), len(T_new_values))
diagonal_mask = np.eye(min_dim, dtype=bool)
off_diagonal_mask = ~np.eye(len(T_orig_values), len(T_new_values), dtype=bool)

diagonal_accs = accuracy_matrix[:min_dim, :min_dim][diagonal_mask]
off_diagonal_accs = accuracy_matrix[off_diagonal_mask]

# Remove NaN values
diagonal_accs = diagonal_accs[~np.isnan(diagonal_accs)]
off_diagonal_accs = off_diagonal_accs[~np.isnan(off_diagonal_accs)]

print(f"\n3. DIAGONAL vs OFF-DIAGONAL ANALYSIS:")
print(f"   • Diagonal (T_orig = T_new) mean accuracy: {np.mean(diagonal_accs):.2f}%")
print(f"   • Off-diagonal (T_orig ≠ T_new) mean accuracy: {np.mean(off_diagonal_accs):.2f}%")
print(f"   • Improvement from T-scaling: {np.mean(off_diagonal_accs) - np.mean(diagonal_accs):+.2f}%")

# Analysis 4: T-scaling directions
print(f"\n4. T-SCALING DIRECTION ANALYSIS:")
scaling_up_improvements = []
scaling_down_improvements = []

for i, orig_T in enumerate(T_orig_values):
    # Find diagonal entry (T_orig = T_new)
    if orig_T in T_new_values:
        orig_idx = T_new_values.index(orig_T)
        orig_acc = accuracy_matrix[i, orig_idx]
        
        if not np.isnan(orig_acc):
            # Scaling up (T_new > T_orig)
            for j, new_T in enumerate(T_new_values):
                if new_T > orig_T and not np.isnan(accuracy_matrix[i, j]):
                    improvement = accuracy_matrix[i, j] - orig_acc
                    scaling_up_improvements.append(improvement)
            
            # Scaling down (T_new < T_orig)
            for j, new_T in enumerate(T_new_values):
                if new_T < orig_T and not np.isnan(accuracy_matrix[i, j]):
                    improvement = accuracy_matrix[i, j] - orig_acc
                    scaling_down_improvements.append(improvement)

if scaling_up_improvements:
    print(f"   • Scaling up (T_new > T_orig) mean improvement: {np.mean(scaling_up_improvements):+.2f}%")
if scaling_down_improvements:
    print(f"   • Scaling down (T_new < T_orig) mean improvement: {np.mean(scaling_down_improvements):+.2f}%")

# Analysis 5: T=255 specific analysis
print(f"\n5. T=255 ANALYSIS:")
if 255 in T_new_values:
    T_255_idx = T_new_values.index(255)
    T_255_accuracies = accuracy_matrix[:, T_255_idx]
    valid_T_255_accs = T_255_accuracies[~np.isnan(T_255_accuracies)]
    
    if len(valid_T_255_accs) > 0:
        print(f"   • Mean accuracy when T_new=255: {np.mean(valid_T_255_accs):.2f}%")
        print(f"   • Best T_orig for T_new=255: T={T_orig_values[np.nanargmax(T_255_accuracies)]} ({np.nanmax(T_255_accuracies):.2f}%)")
        print(f"   • Worst T_orig for T_new=255: T={T_orig_values[np.nanargmin(T_255_accuracies)]} ({np.nanmin(T_255_accuracies):.2f}%)")

print("\n" + "="*80)